<a href="https://colab.research.google.com/github/atomicSteiner/HealthcareSBERT/blob/main/Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
#path for content
path_drive = "/content/drive/MyDrive/NLPFinalProject/"

In [7]:
# Imports
import random
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
import os

In [4]:
#use df
df = pd.read_csv("ohsumed_cleaned_test.csv")
df.head()

,seq_id,medline_ui,mesh_terms,title,abstract
0,54711,88000001,Acetaldehyde/*ME; Buffers; Catalysis; HEPES/PD...,The binding of acetaldehyde to the active site...,"Ribonuclease A was reacted with [1-13C,1,2-14C..."
1,54711,88000002,"Adult; Alcohol, Ethyl/*AN; Breath Tests/*; Hum...",Reductions in breath ethanol readings in norma...,Blood ethanol concentrations were measured seq...
2,54711,88000003,Alcoholism/*PP; Animal; Diprenorphine/PD; Fema...,Does the blockade of opioid receptors influenc...,We have tested whether the opioid antagonists ...
3,54711,88000006,Adult; Alcohol Drinking/*PH; Alcoholism/*BL/CO...,Drinkwatchers--description of subjects and eva...,Clinical examination and measurement of MCV an...
4,54711,88000007,Adult; Alcoholism/*BL; Blood Platelets/*ME; Er...,Platelet affinity for serotonin is increased i...,The kinetics of 3H serotonin platelet uptake w...


In [5]:
# -----------------------------
# Improved MeSH parsing, prepare the data
# In this way we get lists from the initial mesh_terms column
# -----------------------------
def parse_mesh_improved(mesh_str):
    if not isinstance(mesh_str, str):
        return []

    terms = mesh_str.split(';')  # split by semicolon
    clean_terms = []

    for t in terms:
        t = t.strip()            # remove whitespace
        if t == '':
            continue
        # keep only main term (before '/')
        t = t.split('/')[0].strip()
        clean_terms.append(t)

    return clean_terms

df['mesh_terms'] = df['mesh_terms'].apply(parse_mesh_improved)

In [8]:
print(df['mesh_terms'][0])

['Acetaldehyde', 'Buffers', 'Catalysis', 'HEPES', 'Nuclear Magnetic Resonance', 'Phosphates', 'Protein Binding', 'Ribonuclease, Pancreatic', "Support, U.S. Gov't, Non-P.H.S.", "Support, U.S. Gov't, P.H.S.."]


In [2]:
embeddings_baseline = np.load("embeddings_baseline.npy")

In [3]:
# Parametri
N_total = 300
N_mesh = 100
N_nn = 100
N_rand = 100
assert N_mesh + N_nn + N_rand == N_total

random.seed(42)
np.random.seed(42)

In [9]:
# 1) MAP MeSH -> list indici
from collections import defaultdict
mesh_to_idxs = defaultdict(list)
for idx, meshes in enumerate(df['mesh_terms']):
    for m in meshes:
        mesh_to_idxs[m].append(idx)

In [10]:
# 2) Genera coppie MeSH-overlap (evitando duplicati)
mesh_pairs = set()
max_pairs_per_mesh = 20
for mesh, idxs in mesh_to_idxs.items():
    if len(idxs) < 2:
        continue
    sample = random.sample(idxs, min(len(idxs), max_pairs_per_mesh))
    for i in range(len(sample)):
        for j in range(i+1, len(sample)):
            a, b = sample[i], sample[j]
            pair = tuple(sorted((a,b)))
            mesh_pairs.add(pair)
mesh_pairs = list(mesh_pairs)
random.shuffle(mesh_pairs)
mesh_pairs = mesh_pairs[:N_mesh]
print("MeSH pairs:", len(mesh_pairs))

MeSH pairs: 100


In [11]:
# 3) Genera coppie NN (usa embeddings) - per ogni indice prendi nearest neighbor non-identico
nbrs = NearestNeighbors(n_neighbors=6, metric='cosine', algorithm='auto').fit(embeddings_baseline)
distances, indices = nbrs.kneighbors(embeddings_baseline)  # indices[i] contiene i stesso + nearest
nn_pairs = set()
for i in range(len(embeddings_baseline)):
    for neigh_idx in indices[i][1:6]:  # salta il primo (se stesso)
        pair = tuple(sorted((i, int(neigh_idx))))
        # Evita se già MeSH pair (opzionale) - vogliamo varietà
        if pair in mesh_pairs:
            continue
        nn_pairs.add(pair)
nn_pairs = list(nn_pairs)
random.shuffle(nn_pairs)
nn_pairs = nn_pairs[:N_nn]
print("NN pairs:", len(nn_pairs))

NN pairs: 100


In [12]:
# 4) Random pairs (assicurarsi non duplicare con le precedenti)
all_used = set(mesh_pairs) | set(nn_pairs)
rand_pairs = set()
n_doc = len(df)
while len(rand_pairs) < N_rand:
    a, b = np.random.randint(0,n_doc), np.random.randint(0,n_doc)
    if a == b:
        continue
    pair = tuple(sorted((a,b)))
    if pair in all_used or pair in rand_pairs:
        continue
    rand_pairs.add(pair)
rand_pairs = list(rand_pairs)
print("Random pairs:", len(rand_pairs))

Random pairs: 100


In [13]:
# 5) Unire e creare DataFrame di output
pairs_idx = mesh_pairs + nn_pairs + rand_pairs
types = (["mesh"]*len(mesh_pairs)) + (["nn"]*len(nn_pairs)) + (["rand"]*len(rand_pairs))

rows = []
for pid, (i,j) in enumerate(pairs_idx):
    rows.append({
        "pair_id": f"p{pid:04d}",
        "idx_a": i,
        "idx_b": j,
        "text_a": df.loc[i, "abstract"],
        "text_b": df.loc[j, "abstract"],
        "mesh_overlap": bool(set(df.loc[i,'mesh_terms']) & set(df.loc[j,'mesh_terms'])),
        "sample_type": types[pid]
    })
pairs_df = pd.DataFrame(rows)
print(pairs_df.head())
print("Total pairs:", len(pairs_df))


  pair_id   idx_a   idx_b                                             text_a  \
0   p0000  116137  117591  The development of "natural immunity" to homol...   
1   p0001    7180  114696  This study documents the treatment and long-te...   
2   p0002   42052  141387  To study the acute and chronic effects of etha...   
3   p0003  144460  194887  To further characterize the subcellular mechan...   
4   p0004   62583  121036  Three patients underwent single left lung tran...   

                                              text_b  mesh_overlap sample_type  
0  A study was undertaken to determine the incide...          True        mesh  
1  Many patients presenting for treatment of supe...          True        mesh  
2  Antigenic extracts were prepared from Aspergil...          True        mesh  
3  We investigated the effect of tumor necrosis f...          True        mesh  
4  Chronic rejection of the lung in patients with...          True        mesh  
Total pairs: 300


In [14]:
# Salva file con coppie (da distribuire agli annotatori)
pairs_df.to_csv("annotation_pairs.csv", index=False, encoding='utf-8')

Istruzioni per annotatori:

Obiettivo: per ogni coppia di abstract (A, B) assegnare un punteggio di similarità 0–4:

- 4 = Equivalent (stesse informazioni / paraphrase)
- 3 = Highly related (stesso risultato/tema stretto)
- 2 = Related (stesso argomento, ma non stesso risultato)
- 1 = Marginally related (leggera connessione)
- 0 = Not related

Linee guida:
- Leggi entrambi gli abstract per intero.
- Scegli il punteggio che riflette meglio la relazione semantica.
- Se non sei sicuro, usa il valore più basso.
- Compila: pair_id, annotator_id (es. A1), score (0-4), comment (opzionale).

Esempi (impostare alcuni esempi concreti qui).


In [41]:
df_sim_scores = pd.read_csv("similarity_scores_complete.csv")
df_sim_scores.head()

,ID,Similarity_Score
0,0,0
1,1,3
2,2,4
3,3,2
4,4,3


In [42]:
#aggiungo la colonna df_sim_scores["Similarity_Score"] al dataframe pairs_df
pairs_df["human_mean"] = df_sim_scores["Similarity_Score"]
pairs_df.head()

,pair_id,idx_a,idx_b,text_a,text_b,mesh_overlap,sample_type,human_mean
0,p0000,116137,117591,"The development of ""natural immunity"" to homol...",A study was undertaken to determine the incide...,True,mesh,0
1,p0001,7180,114696,This study documents the treatment and long-te...,Many patients presenting for treatment of supe...,True,mesh,3
2,p0002,42052,141387,To study the acute and chronic effects of etha...,Antigenic extracts were prepared from Aspergil...,True,mesh,4
3,p0003,144460,194887,To further characterize the subcellular mechan...,We investigated the effect of tumor necrosis f...,True,mesh,2
4,p0004,62583,121036,Three patients underwent single left lung tran...,Chronic rejection of the lung in patients with...,True,mesh,3


In [43]:
# Salva il nuovo file con la colonna dei punteggi
pairs_df.to_csv("annotation_pairs_with_human_scores.csv", index=False)

Ora che abbiamo le annotazioni umane dobbiamo calcolare quelle dei modelli baseline e finetuned

In [9]:
#importiamo i file necessari:
pairs = pd.read_csv(path_drive + "annotation_pairs_with_human_scores.csv")
baseline_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
ft_model = SentenceTransformer(path_drive + 'fine_tuned_sbert_ohsumed')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
from scipy.stats import pearsonr, spearmanr, ttest_rel

In [11]:
# 1️⃣ Carica le embedding dei due modelli
base = np.load(path_drive + "embeddings_baseline.npy")      # shape: (n, dim)
ft = np.load(path_drive + "fine_tuned_embeddings.npy")

In [12]:
# Assicurati che gli indici siano interi
pairs["idx_a"] = pairs["idx_a"].astype(int)
pairs["idx_b"] = pairs["idx_b"].astype(int)

In [13]:
# 3️⃣ Calcola le similarità per ciascuna coppia
def cosine(a, b):
    return float(cosine_similarity([a], [b])[0][0])

In [14]:
pairs["sim_base"] = [
    cosine(base[i1], base[i2]) for i1, i2 in zip(pairs["idx_a"], pairs["idx_b"])
]
pairs["sim_ft"] = [
    cosine(ft[i1], ft[i2]) for i1, i2 in zip(pairs["idx_a"], pairs["idx_b"])
]

In [15]:
# 4️⃣ Calcola correlazioni con gli score umani
pearson_base = pearsonr(pairs["human_mean"], pairs["sim_base"])
pearson_ft = pearsonr(pairs["human_mean"], pairs["sim_ft"])
spearman_base = spearmanr(pairs["human_mean"], pairs["sim_base"])
spearman_ft = spearmanr(pairs["human_mean"], pairs["sim_ft"])

In [16]:
print(f"Pearson baseline: {pearson_base[0]:.3f}, fine-tuned: {pearson_ft[0]:.3f}")
print(f"Spearman baseline: {spearman_base[0]:.3f}, fine-tuned: {spearman_ft[0]:.3f}")

Pearson baseline: 0.436, fine-tuned: 0.434
Spearman baseline: 0.455, fine-tuned: 0.439


In [17]:
# 5️⃣ Test di significatività
t_stat, p_val = ttest_rel(pairs["sim_ft"], pairs["sim_base"])
print(f"T-test: t = {t_stat:.3f}, p = {p_val:.4f}")

T-test: t = -1.503, p = 0.1340


In [18]:
# 6️⃣ Salva i risultati
pairs.to_csv("evaluation_similarity_results.csv", index=False)

ora proviamo altre metriche, iniziamo con auc

In [19]:
from sklearn.metrics import roc_auc_score, roc_curve, mean_squared_error, mean_absolute_error

# Crea etichette binarie dai punteggi umani
threshold = 3
pairs['label'] = (pairs['human_mean'] >= threshold).astype(int)

# Calcola AUC
auc_base = roc_auc_score(pairs['label'], pairs['sim_base'])
auc_ft = roc_auc_score(pairs['label'], pairs['sim_ft'])

print(f"AUC baseline: {auc_base:.3f}, fine-tuned: {auc_ft:.3f}")


AUC baseline: 0.742, fine-tuned: 0.734


In [20]:
mse_base = mean_squared_error(pairs['human_mean'], pairs['sim_base'])
mse_ft = mean_squared_error(pairs['human_mean'], pairs['sim_ft'])

mae_base = mean_absolute_error(pairs['human_mean'], pairs['sim_base'])
mae_ft = mean_absolute_error(pairs['human_mean'], pairs['sim_ft'])

print(f"MSE baseline: {mse_base:.4f}, fine-tuned: {mse_ft:.4f}")
print(f"MAE baseline: {mae_base:.4f}, fine-tuned: {mae_ft:.4f}")


MSE baseline: 5.6471, fine-tuned: 5.6804
MAE baseline: 2.2036, fine-tuned: 2.2081
